# The goal is to create chat in Gradio where I can select the type of the assistant. It will use Ollama gpt-oss.

In [3]:
import gradio as gr
from openai import OpenAI

In [22]:
OLLAMA_URL = "http://localhost:11434/v1"
ollama = OpenAI(base_url=OLLAMA_URL,api_key="ollama")

In [25]:
!ollama pull gpt-oss

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest 
pulling e7b273f96360:  48% ▕████████          ▏ 6.6 GB/ 13 GB                  pulling manifest 
pulling e7b273f96360:  48% ▕████████          ▏ 6.6 GB/ 13 GB                  pulling manifest 
pulling e7b273f96360:  48% ▕████████          ▏ 6.6 GB/ 13 GB                  pulling manifest 
pulling e7b273f96360:  48% ▕████████          ▏ 6.6 GB/ 13 GB                  pulling manifest 
pulling e7b273f96360:  48% ▕████████          ▏ 6.6 GB/ 13 GB                  pulling manifest 
pulling e7b273f96360:  48% ▕████████          ▏ 6.6 GB/ 13 GB                  pulling manifest 
pulling e7b273f96360:  48% ▕████████          ▏ 6.6 GB/ 13 GB                  pulling manifest 
pulling e7b273f96360:  48% ▕████████          ▏ 6.6 GB/ 13 GB                  pulling manifest 
pulling 

In [24]:
!ollama list

]11;?\NAME                ID              SIZE      MODIFIED    
llama3.2:latest     a80c4f17acd5    2.0 GB    7 days ago     
deepseek-r1:1.5b    e0979632db5a    1.1 GB    2 weeks ago    


In [26]:
assistants = {
    "funny": "You are a funny assistant that always makes jokes.",
    "serious": "You are a serious assistant that always gives straight answers. Never  tells jokes.",
    "scientist": "You are a science assistant that always gives detailed explanations.",
    "arrogant": "You are an arrogant assistant that always thinks it is right."
}

In [20]:
def stream_response(history, assistant_type):
    system_prompt = assistants[assistant_type]

    messages = [{"role": "system", "content": system_prompt}] + history

    stream = ollama.chat.completions.create(
        model="gpt-oss",
        messages=messages,
        stream=True
    )

    partial = ""

    for chunk in stream:
        delta = chunk.choices[0].delta.content
        if delta:
            partial += delta
            yield partial

In [17]:
def chat(message, history, assistant_type):
    if history is None:
        history = []

    # add user message
    history.append({
        "role": "user",
        "content": message
    })

    # placeholder assistant message
    history.append({
        "role": "assistant",
        "content": ""
    })

    # stream tokens
    for partial in stream_response(history[:-1], assistant_type):
        history[-1]["content"] = partial
        yield history, history

In [27]:
def clear_chat():
    return [], []
# -----------------------------
# UI
# -----------------------------
with gr.Blocks(title="Dynamic Assistant Chat") as demo:
    gr.Markdown("# Dynamic Assistant Chat")
    gr.Markdown("Switch assistant type anytime without losing chat history.")

    with gr.Row():
        assistant_selector = gr.Dropdown(
            choices=list(assistants.keys()),
            value="serious",
            label="Assistant Type"
        )

        clear_btn = gr.Button("Clear Chat")

    chatbot = gr.Chatbot(
        # type="messages",
        height=500,
        label="Conversation"
    )

    state = gr.State([])

    with gr.Row():
        msg = gr.Textbox(
            placeholder="Type your message...",
            scale=8
        )
        send = gr.Button("Send", scale=1)

    send.click(
        fn=chat,
        inputs=[msg, state, assistant_selector],
        outputs=[chatbot, state]
    )

    msg.submit(
        fn=chat,
        inputs=[msg, state, assistant_selector],
        outputs=[chatbot, state]
    )

    clear_btn.click(
        fn=clear_chat,
        outputs=[chatbot, state]
    )

demo.launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.
